# 01 · Train-grade predictor (ML)
**Problem:** a train's lab assay comes back 24-48 h after it departs the train load-out (TLO).
By then the ore is at Herb Elliott Port and often already on a vessel. Controllers only have
two in-the-moment signals, and both are biased:
* the **block model** grade of the stockpiles they reclaimed (Solomon over-estimates Fe by ~0.45%)
* the **cross-belt analyser** (PGNAA), which drifts between calibrations

**Model:** gradient-boosted regressor that predicts the *lab* Fe / SiO2 / Al2O3 of a train **at departure**.
It uses analyser statistics, the reclaim-weighted block-model grade, days since analyser calibration,
and each hub's recent lab-vs-sensor bias (only from lab results that had returned *before* the train
departed, so there is no leakage).

Source: `jack_freeman_catalog.fmg_ore_blending.train_features` (Lakeflow gold). Tracked in MLflow,
registered to Unity Catalog as `train_grade_predictor` with alias `@champion`.

In [1]:
import os, warnings
os.environ["GRPC_VERBOSITY"] = "ERROR"
os.environ["GLOG_minloglevel"] = "2"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore")
os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "fevm-serverless-stable-1s8h43")
try:
    spark  # noqa: F821  (defined on Databricks)
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

CATALOG, SCHEMA = "jack_freeman_catalog", "fmg_ore_blending"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.train_grade_predictor"
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/jack.freeman@databricks.com/fmg-ore-blending-train-grade")

df = spark.table(f"{CATALOG}.{SCHEMA}.train_features").toPandas()
df["depart_ts"] = pd.to_datetime(df["depart_ts"])
print(df.shape, "| lab status:", df.lab_status.value_counts().to_dict())
df[["train_id", "hub_code", "product_code", "depart_ts", "bm_fe", "xba_fe", "lab_fe",
    "hub_bm_bias_fe", "hub_xba_bias_fe", "days_since_cal"]].tail(5)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


(1354, 44) | lab status: {'RETURNED': 1325, 'AWAITING_LAB': 23, 'QAQC_REJECTED': 6}


,train_id,hub_code,product_code,depart_ts,bm_fe,xba_fe,lab_fe,hub_bm_bias_fe,hub_xba_bias_fe,days_since_cal
1349,T001350,ELI,FB,2026-11-02 19:50:07,58.205724,57.897552,NaN,-0.187902,0.034778,10
1350,T001351,CB,FB,2026-11-03 03:36:20,58.227335,58.192770,NaN,-0.230822,-0.146716,12
1351,T001355,CC,FB,2026-11-03 03:21:30,58.061510,58.080629,NaN,-0.086722,-0.147421,8
1352,T001359,SOL,FB,2026-11-03 03:56:23,58.295715,57.815099,NaN,-0.428180,0.191312,4
1353,T001363,ELI,FB,2026-11-03 03:33:46,58.412619,58.139321,NaN,-0.191715,0.028189,11


In [2]:
TARGETS = ["fe", "sio2", "al2o3"]
HUBS = ["CB", "CC", "ELI", "SOL"]
NUM = ([f"xba_{e}" for e in ["fe", "sio2", "al2o3", "p", "loi"]] + ["xba_fe_std", "xba_n"]
       + [f"bm_{e}" for e in ["fe", "sio2", "al2o3", "p", "loi"]]
       + [f"hub_xba_bias_{e}" for e in TARGETS] + [f"hub_bm_bias_{e}" for e in TARGETS]
       + ["hub_bias_n", "days_since_cal", "n_stockpiles", "max_stockpile_share", "moisture_pct", "hour_of_day"])


def featurise(d: pd.DataFrame) -> pd.DataFrame:
    X = d[NUM].astype(float).copy()
    for h in HUBS:
        X[f"hub_{h}"] = (d["hub_code"] == h).astype(float)
    X["is_kf"] = (d["product_code"] == "KF").astype(float)
    # physically-motivated features: analyser corrected by its recent bias
    for e in TARGETS:
        X[f"xba_corr_{e}"] = X[f"xba_{e}"] + X[f"hub_xba_bias_{e}"].fillna(0)
        X[f"bm_corr_{e}"] = X[f"bm_{e}"] + X[f"hub_bm_bias_{e}"].fillna(0)
    return X


lab = df[df.has_lab].sort_values("depart_ts")
cut = lab.depart_ts.max() - pd.Timedelta(days=21)          # time-based split: last 3 weeks = test
train, test = lab[lab.depart_ts <= cut], lab[lab.depart_ts > cut]
Xtr, Xte = featurise(train), featurise(test)
print(f"train {len(train)} trains (to {cut:%Y-%m-%d}) | test {len(test)} trains")

train 1012 trains (to 2026-10-11) | test 313 trains


### Baselines vs model (test set = last 21 days, never seen in training)

In [3]:
mlflow.sklearn.autolog(disable=True)
models, rows, q90 = {}, [], {}
with mlflow.start_run(run_name="hgb_train_grade_v1") as run:
    for e in TARGETS:
        m = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04, max_leaf_nodes=15,
                                          min_samples_leaf=20, l2_regularization=1.0, random_state=7)
        m.fit(Xtr, train[f"lab_{e}"])
        models[e] = m
        pred = m.predict(Xte)
        resid = np.abs(test[f"lab_{e}"].values - pred)
        r = dict(element=e,
                 mae_block_model=mean_absolute_error(test[f"lab_{e}"], test[f"bm_{e}"]),
                 mae_analyser_raw=mean_absolute_error(test[f"lab_{e}"], test[f"xba_{e}"]),
                 mae_analyser_bias_corrected=mean_absolute_error(test[f"lab_{e}"], Xte[f"xba_corr_{e}"]),
                 mae_model=float(resid.mean()), p90_abs_err_model=float(np.quantile(resid, 0.9)))
        q90[e] = float(np.quantile(resid, 0.9))  # honest band: p90 abs error on held-out weeks
        rows.append(r)
        for k, v in r.items():
            if k != "element":
                mlflow.log_metric(f"{e}_{k}", v)
    res = pd.DataFrame(rows).set_index("element").round(3)
    mlflow.log_params(dict(model="HistGradientBoostingRegressor", max_iter=400, learning_rate=0.04,
                           split="time-based, last 21 days test", n_train=len(train), n_test=len(test),
                           n_features=Xtr.shape[1]))
    mlflow.log_table(res.reset_index(), "test_metrics.json")
    run_id = run.info.run_id
print("MLflow run:", run_id)
res

🏃 View run hgb_train_grade_v1 at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429/runs/76ececcdfd10422c950c4f64de669d86
🧪 View experiment at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429


MLflow run: 76ececcdfd10422c950c4f64de669d86


,mae_block_model,mae_analyser_raw,mae_analyser_bias_corrected,mae_model,p90_abs_err_model
element,,,,,
fe,0.249,0.172,0.142,0.111,0.238
sio2,0.167,0.141,0.119,0.092,0.187
al2o3,0.063,0.051,0.049,0.038,0.083


### Off-spec detection at departure (vs waiting 24-48 h for the lab)
A train is flagged off-spec if the predicted lab Fe is below the product's typical Fe.

In [4]:
typ = test.product_code.map({"FB": 58.0, "KF": 56.0})
actual_low = test.lab_fe < typ
for name, p in [("block model (today)", test.bm_fe.values), ("analyser raw", test.xba_fe.values),
                ("model", models["fe"].predict(Xte))]:
    flag = p < typ
    tp = int((flag & actual_low).sum()); fp = int((flag & ~actual_low).sum()); fn = int((~flag & actual_low).sum())
    print(f"{name:22s} recall={tp / max(1, tp + fn):.0%} precision={tp / max(1, tp + fp):.0%}  "
          f"(caught {tp} of {tp + fn} below-typical trains, {fp} false alarms)")
with mlflow.start_run(run_id=run_id):
    flag = models["fe"].predict(Xte) < typ
    mlflow.log_metric("offspec_recall_model", float((flag & actual_low).sum() / max(1, actual_low.sum())))
    mlflow.log_metric("offspec_recall_block_model", float(((test.bm_fe < typ) & actual_low).sum() / max(1, actual_low.sum())))

block model (today)    recall=11% precision=93%  (caught 14 of 131 below-typical trains, 1 false alarms)
analyser raw           recall=66% precision=81%  (caught 86 of 131 below-typical trains, 20 false alarms)
model                  recall=71% precision=87%  (caught 93 of 131 below-typical trains, 14 false alarms)


🏃 View run hgb_train_grade_v1 at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429/runs/76ececcdfd10422c950c4f64de669d86
🧪 View experiment at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429


In [5]:
by_hub = test.assign(pred_fe=models["fe"].predict(Xte)).groupby("hub_code").apply(
    lambda g: pd.Series(dict(trains=len(g), mae_block_model=(g.bm_fe - g.lab_fe).abs().mean(),
                             mae_model=(g.pred_fe - g.lab_fe).abs().mean(),
                             bias_block_model=(g.bm_fe - g.lab_fe).mean(), bias_model=(g.pred_fe - g.lab_fe).mean())))
by_hub.round(3)

,trains,mae_block_model,mae_model,bias_block_model,bias_model
hub_code,,,,,
CB,84.0,0.242,0.115,0.237,0.057
CC,83.0,0.109,0.108,0.081,-0.047
ELI,63.0,0.191,0.109,0.174,-0.015
SOL,83.0,0.441,0.110,0.441,0.017


### Package all three element models as one pyfunc and register to Unity Catalog

In [6]:
class TrainGradePredictor(mlflow.pyfunc.PythonModel):
    """Predicts lab Fe/SiO2/Al2O3 of a train at departure, with a 90% error band."""

    def __init__(self, models, q90, hubs):
        self.models, self.q90, self.hubs = models, q90, hubs

    def _featurise(self, d):
        X = d[NUM].astype(float).copy()
        for h in self.hubs:
            X[f"hub_{h}"] = (d["hub_code"] == h).astype(float)
        X["is_kf"] = (d["product_code"] == "KF").astype(float)
        for e in TARGETS:
            X[f"xba_corr_{e}"] = X[f"xba_{e}"] + X[f"hub_xba_bias_{e}"].fillna(0)
            X[f"bm_corr_{e}"] = X[f"bm_{e}"] + X[f"hub_bm_bias_{e}"].fillna(0)
        return X

    def predict(self, context, model_input, params=None):
        X = self._featurise(model_input)
        out = pd.DataFrame(index=model_input.index)
        for e, m in self.models.items():
            out[f"pred_{e}"] = m.predict(X)
            out[f"pred_{e}_band"] = self.q90[e]
        return out


inp = test[["hub_code", "product_code"] + NUM].head(20)
pm = TrainGradePredictor(models, q90, HUBS)
sig = infer_signature(inp, pm.predict(None, inp))
with mlflow.start_run(run_id=run_id):
    info = mlflow.pyfunc.log_model(name="model", python_model=pm, signature=sig, input_example=inp.head(3),
                                   registered_model_name=MODEL_NAME,
                                   pip_requirements=["scikit-learn==1.9.1", "pandas", "numpy"])
from mlflow import MlflowClient
c = MlflowClient()
ver = info.registered_model_version
c.set_registered_model_alias(MODEL_NAME, "champion", ver)
c.update_registered_model(MODEL_NAME, description=(
    "Predicts a train's lab assay (Fe, SiO2, Al2O3) at TLO departure from analyser stats, "
    "reclaim-weighted block model and hub bias known at departure. Trained on Lakeflow gold "
    f"train_features. Test MAE Fe: model {res.loc['fe','mae_model']} vs block model {res.loc['fe','mae_block_model']}."))
print(f"Registered {MODEL_NAME} version {ver} -> @champion")
print("q90 error band:", {k: round(v, 3) for k, v in q90.items()})

/Users/jack.freeman/demos/fortescue-ore-blending/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


🔗 View Logged Model at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429/models/m-d3ef38536ac44ab3aae0c7e76618f7ef


2026/11/03 20:55:14 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/11/03 20:55:14 INFO mlflow.pyfunc: Validating input example against model signature


2026/11/03 20:55:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'jack_freeman_catalog.fmg_ore_blending.train_grade_predictor' already exists. Creating a new version of this model...


🔗 Created version '4' of model 'jack_freeman_catalog.fmg_ore_blending.train_grade_predictor': https://fevm-serverless-stable-1s8h43.cloud.databricks.com/explore/data/models/jack_freeman_catalog/fmg_ore_blending/train_grade_predictor/version/4


🏃 View run hgb_train_grade_v1 at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429/runs/76ececcdfd10422c950c4f64de669d86
🧪 View experiment at: https://fevm-serverless-stable-1s8h43.cloud.databricks.com/ml/experiments/4028418759541429


Registered jack_freeman_catalog.fmg_ore_blending.train_grade_predictor version 4 -> @champion
q90 error band: {'fe': 0.238, 'sio2': 0.187, 'al2o3': 0.083}
